# MJX 05 — Parallel Rollouts & Domain Randomization

### Lab Description

Two ideas turn GPU simulation into a robot-learning tool:

1. **Parallel rollouts** — run many environments at once with different initial conditions by `jax.vmap`-ing over the data.
2. **Domain randomization (DR)** — also randomize the *physics* (mass, friction, ...) across environments. A policy trained over a *distribution* of dynamics never overfits one exact model, so it transfers better to real hardware — this narrows the **sim-to-real gap**.

We slide objects on a plane and show how randomizing **friction** spreads out the stopping distance, both as statistics and as a video.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Batch many rollouts over different initial speeds with `jax.vmap`
- Randomize a physics parameter (`geom_friction`) per environment
- Understand MuJoCo's elementwise-max friction combination rule
- Visualize the resulting distribution of outcomes

### Set up the sliding-object scene

We build a sphere-on-plane scene and move it to MJX. Note the comment on the floor friction: MuJoCo combines the friction of two contacting geoms by taking the **elementwise maximum**, so we keep the floor friction (`0.05`) *below* the randomized range `[0.2, 1.5]`; otherwise the floor would clamp the ball's friction and hide the effect.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jp
import mujoco
from mujoco import mjx

print("JAX devices:", jax.devices())

MJCF = """
<mujoco model="slide">
  <option timestep="0.004"/>
  <worldbody>
    <!-- Keep floor friction below the randomized range [0.2, 1.5]. MuJoCo
         combines contact friction as the elementwise MAX of the two geoms,
         so a high floor value would override (clamp) the ball's friction. -->
    <geom name="floor" type="plane" size="20 20 0.1" friction="0.05 0.005 0.0001"/>
    <body name="ball" pos="0 0 0.1">
      <freejoint/>
      <geom name="ball" type="sphere" size="0.1" friction="0.5 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(MJCF)
mx = mjx.put_model(model)
N_STEPS = 400
ball_geom = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "ball")

### 1. Parallel rollouts — randomized initial speed

Same physics, different initial horizontal velocity per environment. `rollout_speed` scans the physics for one env; `jax.vmap` runs 512 of them at once. We plot a few trajectories — faster launches travel farther before friction stops them.

In [ ]:
def rollout_speed(vx):
    dx = mjx.make_data(mx).replace(qvel=jp.zeros(model.nv).at[0].set(vx))
    def body(dx, _):
        dx = mjx.step(mx, dx)
        return dx, dx.qpos[0]
    dx, xs = jax.lax.scan(body, dx, None, length=N_STEPS)
    return xs

vxs = jp.linspace(1.0, 5.0, 512)
trajs = jax.jit(jax.vmap(rollout_speed))(vxs)
trajs.block_until_ready()

plt.figure(figsize=(7, 3))
for i in range(0, 512, 64):
    plt.plot(np.asarray(trajs[i]), label=f"vx={float(vxs[i]):.1f}")
plt.xlabel("step"); plt.ylabel("ball x (m)"); plt.title("512 parallel rollouts (varying initial speed)")
plt.legend(fontsize=7); plt.grid(True); plt.show()

### 2. Domain randomization — randomized friction

Now we fix the launch speed but **randomize the friction** per environment by batching the model field `geom_friction`. `jax.vmap(make_model)` produces 2048 model variants; `jax.vmap(stopping_x)` rolls each out and returns where the ball stops.

In [ ]:
N_ENVS = 2048
key = jax.random.PRNGKey(0)
# Randomize the tangential friction of the ball geom in [0.2, 1.5].
fr = jax.random.uniform(key, (N_ENVS,), minval=0.2, maxval=1.5)
base_fric = mx.geom_friction  # (ngeom, 3)

def make_model(f):
    new_fric = base_fric.at[ball_geom, 0].set(f)
    return mx.tree_replace({"geom_friction": new_fric})

models = jax.vmap(make_model)(fr)  # batched MJX model

def stopping_x(mxi):
    dx = mjx.make_data(mxi).replace(qvel=jp.zeros(model.nv).at[0].set(4.0))
    def body(dx, _):
        dx = mjx.step(mxi, dx)
        return dx, None
    dx, _ = jax.lax.scan(body, dx, None, length=N_STEPS)
    return dx.qpos[0]

final_x = jax.jit(jax.vmap(stopping_x))(models)
final_x.block_until_ready()
final_x = np.asarray(final_x)
print(f"{N_ENVS} envs, fixed vx=4.0, friction in [0.2,1.5]")
print(f"stopping x: mean={final_x.mean():.2f} m, min={final_x.min():.2f}, max={final_x.max():.2f}")

### Plot the outcome distribution

The scatter shows stopping distance vs. friction (monotonic: more friction → shorter slide); the histogram shows the spread of outcomes the policy would have to be robust to.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(np.asarray(fr), final_x, s=3, alpha=0.4)
axes[0].set_xlabel("friction"); axes[0].set_ylabel("stopping x (m)"); axes[0].set_title("friction vs stopping distance")
axes[1].hist(final_x, bins=40)
axes[1].set_xlabel("stopping x (m)"); axes[1].set_ylabel("count"); axes[1].set_title("outcome distribution under DR")
plt.tight_layout(); plt.show()

### 3. See it — friction spreads the stopping distance

The histogram is abstract, so here is the same idea on screen. Five flat **pucks** launch with the *same* speed but get *different* friction. Higher friction stops a puck sooner, so they fan out across the grid. We use sliding pucks (not the rolling ball) so friction maps cleanly to stopping distance, `d ≈ v² / (2·μ·g)`.

| colour | friction |
|---|---|
| red | 0.2 |
| orange | 0.5 |
| green | 0.8 |
| blue | 1.1 |
| purple | 1.5 |

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import imageio

fr_vals = [0.2, 0.5, 0.8, 1.1, 1.5]
lanes   = [-2, -1, 0, 1, 2]
cols    = ["1 0.3 0.3 1", "1 0.7 0.2 1", "0.3 0.8 0.3 1", "0.3 0.6 1 1", "0.7 0.4 1 1"]
bodies = "".join(
    f'<body name="p{i}" pos="0 {y} 0.05"><freejoint/>'
    f'<geom type="box" size="0.12 0.12 0.05" rgba="{c}" friction="{f} 0.005 0.0001"/></body>'
    for i, (f, y, c) in enumerate(zip(fr_vals, lanes, cols)))

SCENE = f"""
<mujoco>
  <option timestep="0.004"/>
  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.8 0.8 0.8" rgb2="0.6 0.6 0.6" width="512" height="512"/>
    <material name="grid" texture="grid" texrepeat="20 20" reflectance="0.05"/>
  </asset>
  <worldbody>
    <geom name="floor" type="plane" size="20 20 0.1" material="grid" friction="0.05 0.005 0.0001"/>
    {bodies}
  </worldbody>
</mujoco>
"""
vis_model = mujoco.MjModel.from_xml_string(SCENE)
vis_data = mujoco.MjData(vis_model)
for i in range(len(fr_vals)):
    vis_data.qvel[vis_model.jnt_dofadr[i]] = 4.0  # identical launch speed for all pucks
mujoco.mj_forward(vis_model, vis_data)

cam = mujoco.MjvCamera()
cam.lookat[:] = [2.0, 0.0, 0.0]; cam.distance = 6.5; cam.elevation = -55; cam.azimuth = 90

os.makedirs("output/videos", exist_ok=True)
renderer = mujoco.Renderer(vis_model, height=360, width=480)
frames = []
for s in range(500):
    mujoco.mj_step(vis_model, vis_data)
    if s % 4 == 0:
        renderer.update_scene(vis_data, camera=cam)
        frames.append(renderer.render())
renderer.close()
imageio.mimsave("output/videos/mjx05_friction_spread.mp4", frames, fps=30)
print("stopping x by friction:",
      {f: round(float(vis_data.qpos[vis_model.jnt_qposadr[i]]), 2) for i, f in enumerate(fr_vals)})

### Watch the pucks fan out

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx05_friction_spread.mp4")

## Conclusions

You ran thousands of randomized rollouts in parallel and saw how domain randomization produces a *distribution* of outcomes. Training a policy across this whole distribution — rather than one fixed friction — is what makes it robust on hardware whose true friction is unknown. Module C (MJX 06–09) trains such policies with RL.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT